# One Tool, Three Uses: AutoGluon MultiModalPredictor

**MIDAS AI in Research Handbook — Multimodal Learning with AutoGluon**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/autogluon_multimodal_demo.ipynb)

This notebook demonstrates AutoGluon's `MultiModalPredictor` across three different use cases: text only, images only, and text combined with structured tabular features. The core insight is that the calling interface stays the same regardless of what you feed it — AutoGluon detects the data types and routes each column to the appropriate pretrained model automatically.

The three sections build on each other:

| Section | Data type | Dataset | Task |
|---|---|---|---|
| Part 1 | Text only | Stanford SST-2 | Sentiment classification |
| Part 2 | Images only | Fashion-MNIST | Clothing category classification |
| Part 3 | Text + tabular | Amazon Reviews | Star rating prediction |

If you have not already worked through the tabular tutorial in Chapter 17, it is worth doing that first. This notebook assumes you are familiar with the split-fit-evaluate pattern and focuses on what changes when you move beyond structured columns.


## Before You Start: Check for GPU

**Part 2 (image classification) requires a GPU.** Parts 1 and 3 will work on CPU, though they will be slower.

In Colab: **Runtime > Change runtime type > T4 GPU**, then run the cell below to confirm.

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"GPU available: {torch.cuda.get_device_name(0)}")
    print("You are all set for all three sections.")
else:
    print("No GPU detected.")
    print("Parts 1 and 3 will still run, but Part 2 (image classification) will be very slow.")
    print("To enable GPU: Runtime > Change runtime type > T4 GPU")

## Step 1 — Install AutoGluon and Dependencies

This installs the multimodal module along with the datasets library for loading data from Hugging Face. Run once per Colab session.

In [ ]:
!pip install -q autogluon.multimodal datasets Pillow 2>&1 | tail -3
print("Installation complete.")

## Step 2 — Imports


In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings("ignore")

from autogluon.multimodal import MultiModalPredictor
from datasets import load_dataset

print("Ready.")

---

## Part 1: Text Only — Sentiment Classification

The dataset here is Stanford Sentiment Treebank (SST-2), a standard benchmark for sentence-level sentiment classification (Socher et al., 2013). Each row is a short movie review sentence labeled as positive (1) or negative (0).

We use a 600-row subset to keep training time short. The goal is not to beat published benchmarks — it is to show how `MultiModalPredictor` handles a DataFrame that contains only a text column and a label.

In [ ]:
# Load SST-2 from Hugging Face
sst2 = load_dataset("stanfordnlp/sst2", split="train").to_pandas()

# Keep only the columns we need and take a manageable subset
sst2 = sst2[["sentence", "label"]].dropna().sample(600, random_state=42).reset_index(drop=True)

sst2_train = sst2.iloc[:500]
sst2_test  = sst2.iloc[500:]

print(f"Train: {len(sst2_train)} rows, Test: {len(sst2_test)} rows")
sst2_train.head(3)

### Fit

The DataFrame has one text column (`sentence`) and one label column (`label`). AutoGluon recognizes the text column by its content and routes it through a pretrained language model. No feature engineering or tokenization needed on your end.

See Chapter 17 for a deeper explanation of `time_limit` and `presets`.

In [ ]:
predictor_text = MultiModalPredictor(
    label="label",
    problem_type="binary",
    path="ag_sst2_model"
).fit(
    train_data=sst2_train,
    time_limit=120
)

### Evaluate


In [ ]:
results_text = predictor_text.evaluate(sst2_test, metrics=["accuracy", "f1"])
print(results_text)

An accuracy around 0.80–0.88 on 100 test rows is reasonable for a two-minute run on a BERT-style model. The key takeaway is not the exact number — it is that you got a working text classifier from a raw Hugging Face dataset with about ten lines of code.

---

## Part 2: Images Only — Clothing Classification

The dataset is Fashion-MNIST (Xiao et al., 2017), a collection of 28x28 grayscale images of clothing items across 10 categories (T-shirt, trouser, dress, sneaker, and so on). It is a standard benchmark for image classification and is hosted on Hugging Face.

`MultiModalPredictor` for image tasks expects a DataFrame with a column of **file paths** pointing to image files on disk. The cell below loads a subset, saves each image as a PNG, and builds that DataFrame.

In [ ]:
from PIL import Image

# Load Fashion-MNIST from Hugging Face
fmnist = load_dataset("fashion_mnist", split="train").select(range(600))

label_names = [
    "T-shirt", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

img_dir = "fmnist_images"
os.makedirs(img_dir, exist_ok=True)

records = []
for i, item in enumerate(fmnist):
    path = os.path.join(img_dir, f"img_{i}.png")
    item["image"].save(path)
    records.append({"image": path, "label": label_names[item["label"]]})

fmnist_df = pd.DataFrame(records)
fmnist_train = fmnist_df.iloc[:500]
fmnist_test  = fmnist_df.iloc[500:]

print(f"Train: {len(fmnist_train)}, Test: {len(fmnist_test)}")
fmnist_train.head(3)

### Fit

Notice the only difference from Part 1 is the DataFrame content. The `MultiModalPredictor` call is identical. AutoGluon detects that the `image` column contains file paths and routes it through a pretrained vision model (a ResNet or ViT variant, depending on the preset).

This section benefits significantly from a GPU. Training on CPU will work but may take 10–15 minutes.

In [ ]:
predictor_image = MultiModalPredictor(
    label="label",
    problem_type="multiclass",
    path="ag_fmnist_model"
).fit(
    train_data=fmnist_train,
    time_limit=180
)

### Evaluate and Inspect Predictions


In [ ]:
results_image = predictor_image.evaluate(fmnist_test, metrics=["accuracy"])
print(results_image)

In [ ]:
import matplotlib.pyplot as plt

# Show a sample of predictions alongside the actual images
sample = fmnist_test.sample(6, random_state=1).reset_index(drop=True)
preds  = predictor_image.predict(sample)

fig, axes = plt.subplots(1, 6, figsize=(14, 3))
for ax, (_, row), pred in zip(axes, sample.iterrows(), preds):
    img = Image.open(row["image"]).convert("L")
    ax.imshow(img, cmap="gray")
    color = "green" if pred == row["label"] else "red"
    ax.set_title(f"True: {row['label']}\nPred: {pred}", fontsize=8, color=color)
    ax.axis("off")
plt.tight_layout()
plt.show()

Green titles are correct predictions, red are misclassifications. The model distinguishes most categories well with 500 training samples, though visually similar items (Shirt vs T-shirt, Coat vs Pullover) are harder to separate. This is expected behavior and mirrors the challenges in real research image datasets.

---

## Part 3: Text + Tabular — Amazon Review Rating Prediction

This section is the closest to a real research scenario. The dataset is the English subset of the multilingual Amazon Reviews corpus (Keung et al., 2020), available on Hugging Face. Each row represents a product review with a mix of structured fields and free text:

- `review_body`: the written review text
- `review_title`: a short text summary
- `product_category`: the product type (a categorical column)
- `stars`: the rating from 1 to 5 (the prediction target)

The task is to predict the star rating from everything else. This is exactly the kind of data that shows up in survey research, app store analyses, or customer feedback studies: some structured fields, some free-form writing, all in the same table.

In [ ]:
# Load the English Amazon reviews subset
amz = load_dataset("mteb/amazon_reviews_multi", "en", split="train").to_pandas()
amz = amz[["review_body", "review_title", "product_category", "stars"]].dropna()
amz = amz.sample(600, random_state=42).reset_index(drop=True)

amz_train = amz.iloc[:500]
amz_test  = amz.iloc[500:]

print(f"Train: {len(amz_train)}, Test: {len(amz_test)}")
print(f"Label distribution:\n{amz_train['stars'].value_counts().sort_index()}")
amz_train.head(3)

### Fit

The DataFrame now has two text columns (`review_body`, `review_title`), one categorical column (`product_category`), and a numeric label (`stars`). AutoGluon routes each column type to the appropriate model component and fuses the learned representations before making a prediction.

Same call as before.

In [ ]:
predictor_combined = MultiModalPredictor(
    label="stars",
    problem_type="multiclass",
    path="ag_amazon_model"
).fit(
    train_data=amz_train,
    time_limit=180
)

### Evaluate

We evaluate with both accuracy and quadratic weighted kappa (QWK), which is a standard metric for ordinal rating prediction. QWK penalizes predictions proportionally to how far off they are — predicting 2 when the true label is 5 is penalized more than predicting 4.

In [ ]:
results_combined = predictor_combined.evaluate(amz_test, metrics=["accuracy"])
print("Text + tabular combined:")
print(results_combined)

### Compare: Does the Tabular Column Actually Help?

One practical research question is whether adding structured features improves over text alone. A quick way to check is to retrain on only the text columns and compare.

In [ ]:
# Text-only version for comparison
amz_train_text = amz_train[["review_body", "review_title", "stars"]]
amz_test_text  = amz_test[["review_body", "review_title", "stars"]]

predictor_text_only = MultiModalPredictor(
    label="stars",
    problem_type="multiclass",
    path="ag_amazon_textonly_model"
).fit(
    train_data=amz_train_text,
    time_limit=120
)

results_text_only = predictor_text_only.evaluate(amz_test_text, metrics=["accuracy"])
print("Text only:")
print(results_text_only)
print("\nText + tabular combined:")
print(results_combined)

On a small 500-row dataset, the difference between these two versions is often modest. The gain from adding `product_category` may be small because the text already carries most of the predictive signal. But on larger datasets, or in cases where the text is short and the structured fields are more informative, the combined version typically pulls ahead.

This comparison pattern — train once with all features, once without a subset, and compare — is a practical way to check whether each data type is actually contributing.

---

## Summary

All three sections used the same `MultiModalPredictor` interface with different DataFrame structures. The only things that changed were the data and the `problem_type` parameter.

| Section | Columns fed in | AutoGluon routed to |
|---|---|---|
| Text only | 1 text column | Pretrained language model (BERT-family) |
| Images only | 1 image-path column | Pretrained vision model (ResNet / ViT) |
| Text + tabular | 2 text + 1 categorical | Language model + tabular encoder, fused |

From here, the natural next step is Chapter 21, which covers how to validate and interpret these results in a research context — including when a good accuracy number is not the same as a meaningful research finding.